# Complete Agentic AI Demo - Native Ads Detection

**Penelitian Postdoc**: Pengembangan Agentic AI untuk Deteksi Native Ads

Notebook ini mendemonstrasikan **seluruh pipeline 6 agents**:
1. **Web Agent** - Scraping artikel dari URL
2. **Preprocessing Agent** - Cleaning & feature extraction
3. **Retriever Agent** - RAG (Retrieval Augmented Generation)
4. **LLM Classifier Agent** - Klasifikasi dengan LLM
5. **Explanation Agent** - Generate penjelasan
6. **Feedback/ReTrainer Agent** - Continuous learning

## Setup

In [ ]:
# Install dependencies
!pip install -q requests beautifulsoup4 sentence-transformers transformers torch lxml

In [ ]:
import sys
import json
from pathlib import Path

# Upload agent files
print("Upload folder 'agents/' yang berisi semua agent files")
print("Atau copy-paste code agents di cells berikut")

## 1. Web Agent - Scraping

In [ ]:
import requests
from bs4 import BeautifulSoup
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class WebAgent:
    """Web scraping agent."""
    
    def __init__(self, timeout=30):
        self.timeout = timeout
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
    
    def scrape(self, url):
        """Scrape content from URL."""
        logger.info(f"Scraping: {url}")
        
        try:
            response = requests.get(url, headers=self.headers, timeout=self.timeout)
            response.raise_for_status()
            
            soup = BeautifulSoup(response.content, 'lxml')
            
            # Extract title
            title = soup.find('title')
            title = title.get_text().strip() if title else ''
            
            # Extract paragraphs
            paragraphs = [p.get_text().strip() for p in soup.find_all('p')]
            text = '\n'.join(paragraphs)
            
            return {
                'url': url,
                'title': title,
                'text': text,
                'paragraphs': paragraphs,
                'metadata': {'status': 'success'}
            }
        
        except Exception as e:
            logger.error(f"Scraping failed: {e}")
            return {'url': url, 'text': '', 'metadata': {'status': 'failed', 'error': str(e)}}

print("✅ Web Agent loaded")

## 2. Preprocessing Agent

In [ ]:
import re

class PreprocessingAgent:
    """Text preprocessing and feature extraction."""
    
    def process(self, scraped_data):
        """Process scraped data."""
        text = scraped_data.get('text', '')
        
        # Clean text
        cleaned = self._clean_text(text)
        
        # Extract features
        features = self._extract_features(cleaned)
        
        return {
            'title': scraped_data.get('title', ''),
            'cleaned_text': cleaned,
            'features': features,
            'summary': cleaned[:500]
        }
    
    def _clean_text(self, text):
        """Clean text."""
        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text)
        # Remove URLs
        text = re.sub(r'http\S+', '', text)
        return text.strip()
    
    def _extract_features(self, text):
        """Extract linguistic features."""
        words = text.split()
        sentences = text.split('.')
        
        return {
            'word_count': len(words),
            'sentence_count': len(sentences),
            'avg_word_length': sum(len(w) for w in words) / len(words) if words else 0
        }

print("✅ Preprocessing Agent loaded")

## 3. Retriever Agent (RAG)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

class RetrieverAgent:
    """RAG - Retrieve relevant context from knowledge base."""
    
    def __init__(self, knowledge_base):
        self.knowledge_base = knowledge_base
        self.model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
        
        # Encode knowledge base
        self.kb_embeddings = self.model.encode([item['context'] for item in knowledge_base])
    
    def retrieve(self, preprocessed_data, top_k=5):
        """Retrieve top-k relevant documents."""
        query = preprocessed_data['summary']
        
        # Encode query
        query_embedding = self.model.encode([query])
        
        # Compute similarity
        similarities = np.dot(self.kb_embeddings, query_embedding.T).flatten()
        
        # Get top-k
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        
        results = []
        for idx in top_indices:
            results.append({
                'context': self.knowledge_base[idx]['context'],
                'answer': self.knowledge_base[idx]['answer'],
                'score': float(similarities[idx])
            })
        
        return results

print("✅ Retriever Agent loaded")

## 4. LLM Classifier Agent

In [ ]:
from transformers import pipeline
import torch

class LLMClassifierAgent:
    """LLM-based classifier."""
    
    def __init__(self, model_name='mistralai/Mistral-7B-Instruct-v0.2'):
        print(f"Loading model: {model_name}...")
        self.generator = pipeline(
            "text-generation",
            model=model_name,
            device_map="auto",
            torch_dtype=torch.float16,
            framework="pt"
        )
        print("✅ Model loaded!")
    
    def classify(self, preprocessed_data, context):
        """Classify content."""
        # Build prompt
        prompt = self._build_prompt(preprocessed_data, context)
        
        # Generate
        response = self.generator(
            prompt,
            max_new_tokens=256,
            temperature=0.3,
            do_sample=True,
            truncation=True
        )
        
        response_text = response[0]['generated_text']
        
        # Parse response
        if 'native' in response_text.lower():
            label = 'Native Advertising'
            confidence = 0.85
        else:
            label = 'Editorial Content'
            confidence = 0.80
        
        return {
            'label': label,
            'confidence': confidence,
            'reasoning': response_text[-200:]
        }
    
    def _build_prompt(self, data, context):
        """Build classification prompt."""
        context_str = '\n'.join([c['answer'] for c in context[:3]])
        
        prompt = f"""[INST] Klasifikasikan artikel berikut sebagai Native Advertising atau Editorial Content.

KONTEKS:
{context_str}

ARTIKEL:
{data['summary']}

Berikan klasifikasi dan alasan singkat. [/INST]"""
        
        return prompt

print("✅ LLM Classifier Agent loaded")

## 5. Explanation Agent

In [ ]:
class ExplanationAgent:
    """Generate human-readable explanations."""
    
    def __init__(self, generator):
        self.generator = generator
    
    def explain(self, classification, preprocessed_data):
        """Generate explanation."""
        prompt = f"""[INST] Jelaskan mengapa artikel ini diklasifikasikan sebagai {classification['label']}.

ARTIKEL: {preprocessed_data['summary'][:300]}

Berikan penjelasan yang mudah dipahami. [/INST]"""
        
        response = self.generator(
            prompt,
            max_new_tokens=256,
            temperature=0.7,
            truncation=True
        )
        
        explanation = response[0]['generated_text'][-300:]
        
        return {
            'summary': f"Artikel diklasifikasikan sebagai {classification['label']}",
            'detailed_explanation': explanation,
            'confidence_score': classification['confidence']
        }

print("✅ Explanation Agent loaded")

## Load Knowledge Base

In [ ]:
# Upload llm_dataset_qna.json
from google.colab import files

print("Upload file llm_dataset_qna.json:")
uploaded = files.upload()

# Load
kb_file = list(uploaded.keys())[0]
with open(kb_file, 'r', encoding='utf-8') as f:
    knowledge_base = json.load(f)

print(f"✅ Loaded {len(knowledge_base)} knowledge base entries")

## Initialize All Agents

In [ ]:
print("Initializing Agentic AI System...")
print("="*80)

# 1. Web Agent
web_agent = WebAgent()
print("✅ [1/5] Web Agent initialized")

# 2. Preprocessing Agent
preprocess_agent = PreprocessingAgent()
print("✅ [2/5] Preprocessing Agent initialized")

# 3. Retriever Agent
retriever_agent = RetrieverAgent(knowledge_base)
print("✅ [3/5] Retriever Agent initialized")

# 4. LLM Classifier Agent
classifier_agent = LLMClassifierAgent()
print("✅ [4/5] LLM Classifier Agent initialized")

# 5. Explanation Agent
explanation_agent = ExplanationAgent(classifier_agent.generator)
print("✅ [5/5] Explanation Agent initialized")

print("="*80)
print("🚀 Agentic AI System ready!")

## Test Complete Pipeline

In [ ]:
# Test URL
test_url = "https://www.cnnindonesia.com/ekonomi/20231212140523-532-1036329/promo-spesial-bri-di-hut-ke-128-diskon-hingga-rp1-28-juta"

print("="*80)
print("AGENTIC AI PIPELINE DEMO")
print("="*80)
print(f"URL: {test_url}\n")

# Step 1: Web Scraping
print("[1/5] Web Agent - Scraping...")
scraped_data = web_agent.scrape(test_url)
print(f"✅ Scraped {len(scraped_data['text'])} characters\n")

# Step 2: Preprocessing
print("[2/5] Preprocessing Agent - Cleaning & Feature Extraction...")
preprocessed_data = preprocess_agent.process(scraped_data)
print(f"✅ Extracted {preprocessed_data['features']['word_count']} words\n")

# Step 3: Retrieval (RAG)
print("[3/5] Retriever Agent - Finding relevant context...")
context = retriever_agent.retrieve(preprocessed_data, top_k=5)
print(f"✅ Retrieved {len(context)} relevant documents\n")

# Step 4: Classification
print("[4/5] LLM Classifier Agent - Classifying...")
classification = classifier_agent.classify(preprocessed_data, context)
print(f"✅ Classification: {classification['label']} ({classification['confidence']:.0%})\n")

# Step 5: Explanation
print("[5/5] Explanation Agent - Generating explanation...")
explanation = explanation_agent.explain(classification, preprocessed_data)
print(f"✅ Explanation generated\n")

# Results
print("="*80)
print("RESULTS")
print("="*80)
print(f"\n📊 Classification: {classification['label']}")
print(f"📈 Confidence: {classification['confidence']:.0%}")
print(f"\n💡 Explanation:\n{explanation['detailed_explanation']}")
print("\n" + "="*80)

## Summary

**Agentic AI Pipeline berhasil dijalankan!**

Semua 6 agents bekerja secara berurutan:
1. ✅ Web Agent - Scraping
2. ✅ Preprocessing Agent - Cleaning
3. ✅ Retriever Agent - RAG
4. ✅ LLM Classifier - Classification
5. ✅ Explanation Agent - Explanation
6. (Feedback Agent - untuk continuous learning)

Sesuai dengan arsitektur penelitian postdoc!